# Lab 07 — Ensemble Models, Built from Scratch

**Class 07.** Guide: `practice/07_ensembles.pdf`. Lecture: `slides/07_ensembles.pdf`. Demos: `01_shallow_models.ipynb`
to `07_real_tables.ipynb`. In class (60 minutes): Parts A to D; at home: Parts E and F.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work.

**The rule of this lab.** An ensemble is a loop around a weak learner. You write the loops with NumPy (the weak learner is
a scikit-learn tree) and compare each with its scikit-learn twin on the **same split**. The notebook prints the difference;
explain it.

In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab07_utils import (
    compare,
    load_heart,
    mae,
    mcc,
    moons_split,
    noisy_features,
    plot_boundaries,
    sine_split,
    ten_points,
    three_members,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore")
pl.Config.set_fmt_str_lengths(60)
plt.rcParams.update({"figure.dpi": 100})

# Part A — The Gallery and a Vote

The running example of the lecture: 300 moons with noise 0.3, a stratified 70/30 split (210 train, 90 test; one test point
is 0.011). Three members of different families (Classes 03, 05, 06) draw three boundaries; a **vote** combines them.

## A1. Hard vote and soft vote

Implement `hard_vote(labels)` (`labels` has shape `(B, n)`, one row of 0/1 predictions per member; return the majority label
of each example, ties go to class 1) and `soft_vote(probas)` (`probas` has shape `(B, n)`, the probability of class 1 of each
member; return the mean probability and the label `mean > 0.5`).

In [ ]:
def hard_vote(labels):
    return (np.asarray(labels).mean(axis=0) >= 0.5).astype(int)


def soft_vote(probas):
    p = np.asarray(probas).mean(axis=0)
    return p, (p > 0.5).astype(int)

The driver fits the three members, draws them next to the soft vote, and compares with `VotingClassifier(voting="soft")`.

In [ ]:
X_tr, X_te, y_tr, y_te = moons_split(0)
members = {name: m.fit(X_tr, y_tr) for name, m in three_members(0).items()}
P = np.array([m.predict_proba(X_te)[:, 1] for m in members.values()])
L = (P > 0.5).astype(int)
p_vote, y_soft = soft_vote(P)
y_hard = hard_vote(L)
twin = VotingClassifier(list(three_members(0).items()), voting="soft").fit(X_tr, y_tr)
y_twin = twin.predict(X_te)
rows = [(name, f"{mcc(y_te, lab):.3f}") for name, lab in zip(members, L, strict=True)]
rows += [("hard vote", f"{mcc(y_te, y_hard):.3f}"), ("soft vote", f"{mcc(y_te, y_soft):.3f}"),
         ("VotingClassifier", f"{mcc(y_te, y_twin):.3f}")]  # fmt: skip
print(pl.DataFrame(rows, schema=["model", "test MCC"], orient="row"))
print(f"soft vote agrees with VotingClassifier on {np.mean(y_soft == y_twin):.1%} of the test points")
print(f"points where the three members disagree: {int(np.sum(L.min(axis=0) != L.max(axis=0)))} of {len(y_te)}")
fig = plot_boundaries({**members, "soft vote": lambda Z: np.mean([m.predict_proba(Z)[:, 1] for m in members.values()], axis=0)},
                      X_tr, y_tr, X_te, y_te)  # fmt: skip
plt.show()

**Questions A1.**

1. What are the test MCC values of the three members, of the hard vote and of the soft vote? Is the vote better than the
   best member? Than the mean member?
2. On how many test points do the members disagree? Why does a vote need that number to be above zero?
3. Replace the three members by three copies of 15-NN. What does the vote give, and what does it say about diversity?

**How we got there.** The mean of the member probabilities is the soft vote; thresholding at 0.5 gives the label. A hard
vote is the mean of the 0/1 labels thresholded at 0.5 (with an odd number of members there is no tie).

**Answers A1.** See the printed table: the vote (MCC 0.781) equals the SVM, beats the logistic model (0.512) and the mean
member (0.735), but it does not beat 15-NN (0.912) on this split, where 15-NN is lucky on 90 points. The members disagree on
18 of 90 points; with zero disagreement all members are the same function and the vote equals each of them. Three copies
of 15-NN vote exactly like one 15-NN: copies add no diversity.


# Part B — Bagging by Hand

A bootstrap sample draws $n$ examples **with replacement**: about $63.2\,\%$ of the distinct examples are in, $36.8\,\%$ are
**out of bag** (OOB). A bagged ensemble trains one deep tree per sample and averages their probabilities; the OOB score
predicts each example only with the trees that did not see it.

## B1. Bootstrap, bagging, OOB MCC

Implement `bootstrap_indices(n, rng)` (`n` indices drawn with replacement), `fit_bagging(X, y, n_trees, seed, max_features=None)`
(returns the list of trees and a boolean array `inbag` of shape `(n_trees, n)`; tree `b` is a `DecisionTreeClassifier`
with `max_features=max_features` and `random_state=seed + b`), `predict_bagging(trees, X)` (the mean probability of class 1)
and `oob_mcc(trees, inbag, X, y)` (MCC of the mean probability of the trees with `inbag[b, i] == False`; skip examples
that are in every sample).

In [ ]:
def bootstrap_indices(n, rng):
    return rng.integers(0, n, size=n)


def fit_bagging(X, y, n_trees=100, seed=0, max_features=None):
    rng = np.random.default_rng(seed)
    trees, inbag = [], np.zeros((n_trees, len(y)), dtype=bool)
    for b in range(n_trees):
        idx = bootstrap_indices(len(y), rng)
        inbag[b, idx] = True
        trees.append(DecisionTreeClassifier(max_features=max_features, random_state=seed + b).fit(X[idx], y[idx]))
    return trees, inbag


def predict_bagging(trees, X):
    return np.mean([t.predict_proba(X)[:, 1] for t in trees], axis=0)


def oob_mcc(trees, inbag, X, y):
    P = np.array([t.predict_proba(X)[:, 1] for t in trees])  # (B, n)
    out = ~inbag
    seen = out.sum(axis=0) > 0
    p = (P * out).sum(axis=0)[seen] / out.sum(axis=0)[seen]
    return mcc(y[seen], p > 0.5)

The driver checks the share of unique examples, then compares your 100 trees with `BaggingClassifier` and
`RandomForestClassifier` on the moons, and sweeps `max_features` on a set with 20 correlated features.

In [ ]:
rng = np.random.default_rng(0)
print(
    "unique share of a bootstrap sample:",
    {n: round(len(np.unique(bootstrap_indices(n, rng))) / n, 3) for n in (10, 210, 10000)},
    "(1 - 1/e = 0.632)",
)
trees, inbag = fit_bagging(X_tr, y_tr, 100, seed=0)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=0).fit(X_tr, y_tr)
rf = RandomForestClassifier(100, oob_score=True, random_state=0).fit(X_tr, y_tr)
single = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
rows = [
    ("one tree", f"{mcc(y_te, single.predict(X_te)):.3f}", "—"),
    (
        "ours, 100 trees",
        f"{mcc(y_te, predict_bagging(trees, X_te) > 0.5):.3f}",
        f"{oob_mcc(trees, inbag, X_tr, y_tr):.3f}",
    ),
    (
        "BaggingClassifier",
        f"{mcc(y_te, bag.predict(X_te)):.3f}",
        f"{mcc(y_tr, bag.oob_decision_function_[:, 1] > 0.5):.3f}",
    ),
    (
        "RandomForestClassifier",
        f"{mcc(y_te, rf.predict(X_te)):.3f}",
        f"{mcc(y_tr, rf.oob_decision_function_[:, 1] > 0.5):.3f}",
    ),
]
print(pl.DataFrame(rows, schema=["model", "test MCC", "OOB MCC"], orient="row"))

Xn, yn = noisy_features(0)
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.3, stratify=yn, random_state=0)
rows = []
for m in (1, 4, 10, 20):
    tr, ib = fit_bagging(Xn_tr, yn_tr, 100, seed=0, max_features=m)
    rows.append(
        (
            m,
            round(oob_mcc(tr, ib, Xn_tr, yn_tr), 3),
            round(mcc(yn_te, predict_bagging(tr, Xn_te) > 0.5), 3),
        )
    )
print(pl.DataFrame(rows, schema=["max_features", "OOB MCC", "test MCC"], orient="row"))

**Questions B1.**

1. Is the unique share close to $1 - 1/e$? Why does it tend to that value?
2. How do your OOB MCC and test MCC compare with the two library twins? Why do they not agree to the last digit?
3. With 20 features, which `max_features` has the highest OOB MCC, and how much better is it than a value of 20 (plain bagging)
   than a smaller one?

**How we got there.** A bootstrap sample is `rng.integers(0, n, n)`; marking `inbag[b, idx]` records which examples tree
`b` saw. The OOB prediction of example $i$ is the mean probability over the trees with `inbag[b, i] == False`, about 37
of 100 trees.

**Answers B1.** The unique share is 0.7 for ten examples on this draw and 0.634 for 10000: the chance of being missed is
$(1 - 1/n)^n \to e^{-1}$, so it tends to 0.632. Our OOB MCC (0.867) estimates the test MCC (0.870) with no hold-out set; the
twins give 0.848 against 0.850 and 0.829 against 0.805, within the noise of 90 points. They differ by the random draws and,
for the forest, by `max_features` (square root of $d$ in `RandomForestClassifier`, all features in bagging). On 20
correlated features a small `max_features` decorrelates the trees, so the average has a lower variance; with all features
every tree picks the same strong features first. Here the OOB MCC is 0.667 at 1, 0.712 at 4, 0.709 at 10 and at 20: the
best value is 4, and its gain over plain bagging is only 0.003 on this set.


# Part C — AdaBoost with Stumps

Boosting trains the members in turn: each **stump** (a tree with one question) is fitted with example weights, and the
examples it gets wrong weigh more for the next one. With labels $s_i = 2y_i - 1 \in \{-1, +1\}$, round $b$ computes the
weighted error $\text{err}_b = \sum_i w_i^{(b)} [\hat f_b(x_i) \ne y_i]$, the member weight
$\beta_b = \tfrac12\ln\frac{1-\text{err}_b}{\text{err}_b}$ and the new weights
$w_i^{(b+1)} = w_i^{(b)} e^{-\beta_b s_i \hat s_b(x_i)} / Z_b$, with $Z_b$ so that they sum to 1.

## C1. The loop

Implement `fit_adaboost(X, y, n_rounds)`: weights start at $1/n$; each round fits
`DecisionTreeClassifier(max_depth=1)` with `sample_weight=w`, computes $\text{err}_b$, $\beta_b$ and the new weights; stop early
if $\text{err}_b = 0$. Return the list of stumps, the array of $\beta_b$ and the list of weight vectors used in each round.
Implement `predict_adaboost(stumps, betas, X)`: the sign of $\sum_b \beta_b \hat s_b(x)$, returned as 0/1 labels.

In [ ]:
def fit_adaboost(X, y, n_rounds=50):
    n, s = len(y), 2 * y - 1
    w = np.full(n, 1.0 / n)
    stumps, betas, weights = [], [], []
    for _ in range(n_rounds):
        stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y, sample_weight=w)
        pred_s = 2 * stump.predict(X) - 1
        err = float(np.sum(w[pred_s != s]))
        if err == 0:
            stumps.append(stump)
            betas.append(5.0)
            weights.append(w.copy())  # a perfect stump: a large finite weight
            break
        beta = 0.5 * np.log((1 - err) / err)
        stumps.append(stump)
        betas.append(beta)
        weights.append(w.copy())
        w = w * np.exp(-beta * s * pred_s)
        w = w / w.sum()
    return stumps, np.array(betas), weights


def predict_adaboost(stumps, betas, X):
    score = sum(b * (2 * st.predict(X) - 1) for st, b in zip(stumps, betas, strict=True))
    return (score > 0).astype(int)

The driver reproduces the ten-point example of the lecture (round 1: $\text{err}_1 = 0.2$, $\beta_1 = 0.693$), then fits
50 rounds on the moons and compares with `AdaBoostClassifier` on the same stumps.

In [ ]:
Xt, yt = ten_points()
st10, be10, w10 = fit_adaboost(Xt, yt, 3)
print("ten points: beta =", np.round(be10, 3), "| weights after round 1:", np.round(w10[1], 4) if len(w10) > 1 else "—")
print("ten points: ensemble right on", int(np.sum(predict_adaboost(st10, be10, Xt) == yt)), "of 10")

stumps, betas, _ = fit_adaboost(X_tr, y_tr, 50)
ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=0).fit(X_tr, y_tr)
ours_te, ref_te = predict_adaboost(stumps, betas, X_te), ada.predict(X_te)
print(
    compare(
        [
            (
                "test MCC",
                f"{mcc(y_te, ours_te):.3f}",
                f"{mcc(y_te, ref_te):.3f}",
                f"{np.mean(ours_te == ref_te):.1%}",
            ),
            ("beta of round 1", f"{betas[0]:.3f}", f"{ada.estimator_weights_[0]:.3f}", "twice"),
        ]
    )
)
rounds = np.arange(1, len(stumps) + 1)
te_curve = [mcc(y_te, predict_adaboost(stumps[:r], betas[:r], X_te)) for r in rounds]
tr_curve = [mcc(y_tr, predict_adaboost(stumps[:r], betas[:r], X_tr)) for r in rounds]
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(rounds, tr_curve, label="train MCC")
ax.plot(rounds, te_curve, label="test MCC")
ax.set(xlabel="rounds $b$", ylabel="MCC")
ax.legend()
plt.show()

**Questions C1.**

1. In the ten-point example, which points gain weight after round 1, and by how much? Is the final vote right on all ten?
2. How close is your ensemble to `AdaBoostClassifier`? The library reports $\beta$ twice as large as yours: why does the
   vote not change?
3. Read the curves: does the test MCC keep rising with more rounds? What would you use to choose the number of rounds?

**How we got there.** The loss of adding a member of weight $\beta$ is $(1-\text{err})e^{-\beta} + \text{err}\,e^{\beta}$; its
minimum is $\beta = \tfrac12\ln\frac{1-\text{err}}{\text{err}}$. A right point is multiplied by $e^{-\beta}$, a wrong one
by $e^{\beta}$, then everything is divided by their sum $Z$.

**Answers C1.** In round 1 the stump $x_1 \le 0.053$ gets points 2 and 4 wrong: their weight goes from 0.10 to 0.25 and the
other eight fall to 0.0625; $Z = 0.8$. The three-round vote is right on all ten points. Our ensemble gives the same
predictions as `AdaBoostClassifier` (agreement 100 %): the library's $\beta$ is $\ln\frac{1-\text{err}}{\text{err}}$,
exactly twice ours, and doubling every weight of a sign vote does not change the sign. The test MCC rises from 0.511 at one
round to 0.785 at 20 and then stays flat, as does the training MCC (0.888): choose the number of rounds on validation data
(Part D, early stopping).

# Part D — Gradient Boosting for Regression

**Gradient boosting** adds trees that fit the **residuals**. Start from $F_0 = \bar y$; in round $b$ fit a small regression
tree $\hat f_b$ to the residuals $r_i = y_i - F_{b-1}(x_i)$ and set $F_b = F_{b-1} + \eta\,\hat f_b$, with the shrinkage
$\eta$ (the learning rate of Class 02).

## D1. Residuals, shrinkage, early stopping

Implement `fit_gbr(X, y, n_trees, eta, depth)` returning `(F0, trees, eta)` (`DecisionTreeRegressor(max_depth=depth)`),
`predict_gbr(model, X, n_trees=None)` (the prediction with the first `n_trees` trees, all if `None`) and
`best_rounds(model, X_val, y_val)` (the number of trees, from 1, with the lowest validation MAE).

In [ ]:
def fit_gbr(X, y, n_trees=100, eta=0.1, depth=2):
    f0 = float(np.mean(y))
    F = np.full(len(y), f0)
    trees = []
    for _ in range(n_trees):
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, y - F)
        F = F + eta * tree.predict(X)
        trees.append(tree)
    return f0, trees, eta


def predict_gbr(model, X, n_trees=None):
    f0, trees, eta = model
    out = np.full(len(X), f0)
    for tree in trees[:n_trees]:
        out = out + eta * tree.predict(X)
    return out


def best_rounds(model, X_val, y_val):
    f0, trees, eta = model
    out, errors = np.full(len(X_val), f0), []
    for tree in trees:
        out = out + eta * tree.predict(X_val)
        errors.append(mae(y_val, out))
    return int(np.argmin(errors)) + 1

The driver fits 100 trees of depth 2 with $\eta = 0.1$ on the noisy sine, compares with `GradientBoostingRegressor`, sweeps
$\eta$ and chooses the number of trees on the validation part. Test targets are the noiseless sine on a longer range.

In [ ]:
Xs_tr, ys_tr, Xs_va, ys_va, Xs_te, ys_te = sine_split(0)
model = fit_gbr(Xs_tr, ys_tr, 100, 0.1, 2)
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0).fit(Xs_tr, ys_tr)
inside = np.abs(Xs_te[:, 0]) <= 3
print(
    compare(
        [
            (
                "test MAE inside [-3, 3]",
                f"{mae(predict_gbr(model, Xs_te)[inside], ys_te[inside]):.4f}",
                f"{mae(gb.predict(Xs_te)[inside], ys_te[inside]):.4f}",
                f"max |diff| {np.abs(predict_gbr(model, Xs_te) - gb.predict(Xs_te)).max():.1e}",
            ),
            (
                "test MAE outside",
                f"{mae(predict_gbr(model, Xs_te)[~inside], ys_te[~inside]):.4f}",
                f"{mae(gb.predict(Xs_te)[~inside], ys_te[~inside]):.4f}",
                "flat outside",
            ),
        ]
    )
)
rows = []
for eta in (1.0, 0.3, 0.1, 0.03):
    m = fit_gbr(Xs_tr, ys_tr, 300, eta, 3)
    k = best_rounds(m, Xs_va, ys_va)
    rows.append(
        (
            eta,
            k,
            round(mae(predict_gbr(m, Xs_te, k)[inside], ys_te[inside]), 3),
            round(mae(predict_gbr(m, Xs_te)[inside], ys_te[inside]), 3),
        )
    )
print(
    pl.DataFrame(rows, schema=["eta", "best trees (validation)", "test MAE at best", "test MAE at 300"], orient="row")
)
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.scatter(Xs_tr[:, 0], ys_tr, s=8, color="#999999", label="train")
ax.plot(Xs_te[:, 0], ys_te, color="#333333", label="sine")
for n in (1, 5, 30, 100):
    ax.plot(Xs_te[:, 0], predict_gbr(model, Xs_te, n), label=f"{n} trees")
ax.legend(fontsize=7)
plt.show()

**Questions D1.**

1. How close is your model to `GradientBoostingRegressor`? Why can they agree to rounding error?
2. How does the best number of trees change with $\eta$, and what happens to the test MAE at 300 trees when $\eta$ is 1?
3. What does the model predict outside $[-3, 3]$, and why?

**How we got there.** For squared loss the negative gradient of the loss with respect to the prediction is the residual
$y - F$, so "fit a tree to the residuals" is a gradient step in function space; $\eta$ scales the step.

**Answers D1.** Both start from the mean, fit depth-2 trees to the same residuals with the same shrinkage and no
subsampling, so the predictions agree exactly here. With $\eta = 1$ the best MAE is 0.215 and the model overfits after 9
trees; with smaller $\eta$ the best MAE is 0.152 to 0.158. A small $\eta$ needs more trees (68 at 0.03, against 6 at 0.3)
and overfits more slowly: at 300 trees the MAE is 0.244 for $\eta = 1$ and 0.177 for $\eta = 0.03$. Outside the training
range the prediction is flat: a sum of trees is a staircase and cannot extrapolate (Class 06).


# Part E — Stacking with Out-of-Fold Predictions (home)

A **stack** feeds the predictions of the members to a meta-learner. If the members predict their own training points, the
meta-learner sees predictions that are too good and trusts the members too much: this is **leakage**. The cure is to
give it **out-of-fold** predictions, where each example is predicted by a member that did not train on it.

## E1. Out-of-fold features and the stack

Implement `out_of_fold(make_model, X, y, k, seed)`: with `StratifiedKFold(k, shuffle=True, random_state=seed)`, fit a fresh
`make_model()` on the other $k-1$ folds and return the probability of class 1 of every example, predicted by the
model that did not see it (an array of shape `(n,)`). Implement `fit_stack(makers, X, y, k, seed, leak=False)`: the
features of the meta-learner (`LogisticRegression`) are the out-of-fold probabilities of every member (the in-fold ones
if `leak=True`); each member is then refitted on all of `X`. Return `(members, meta)`. Implement `predict_stack(stack, X)`.

In [ ]:
def out_of_fold(make_model, X, y, k=5, seed=0):
    p = np.zeros(len(y))
    for tr, va in StratifiedKFold(k, shuffle=True, random_state=seed).split(X, y):
        p[va] = make_model().fit(X[tr], y[tr]).predict_proba(X[va])[:, 1]
    return p


def fit_stack(makers, X, y, k=5, seed=0, leak=False):
    members = [make().fit(X, y) for make in makers]
    if leak:
        Z = np.column_stack([m.predict_proba(X)[:, 1] for m in members])
    else:
        Z = np.column_stack([out_of_fold(make, X, y, k, seed) for make in makers])
    return members, LogisticRegression(C=1.0, max_iter=1000).fit(Z, y)


def predict_stack(stack, X):
    members, meta = stack
    return meta.predict_proba(np.column_stack([m.predict_proba(X)[:, 1] for m in members]))[:, 1]

The driver uses the moons with three members (1-NN, a tree, an SVM) and compares the leaking and the out-of-fold stack
with `StackingClassifier`. The meta-learner is scored by cross-validation on its own features and on the test set.

In [ ]:
makers = [
    lambda: KNeighborsClassifier(1),
    lambda: DecisionTreeClassifier(random_state=0),
    lambda: CalibratedClassifierCV(SVC(random_state=0), method="sigmoid", cv=5, ensemble=False),
]
rows = []
for name, leak in (("stack, in-fold (leak)", True), ("stack, out-of-fold", False)):
    st = fit_stack(makers, X_tr, y_tr, 5, 0, leak=leak)
    rows.append(
        (
            name,
            f"{mcc(y_tr, predict_stack(st, X_tr) > 0.5):.3f}",
            f"{mcc(y_te, predict_stack(st, X_te) > 0.5):.3f}",
            np.round(st[1].coef_[0], 2).tolist(),
        )
    )
sk = StackingClassifier([(f"m{i}", mk()) for i, mk in enumerate(makers)], LogisticRegression(max_iter=1000), cv=5).fit(
    X_tr, y_tr
)
rows.append(
    (
        "StackingClassifier",
        f"{mcc(y_tr, sk.predict(X_tr)):.3f}",
        f"{mcc(y_te, sk.predict(X_te)):.3f}",
        np.round(sk.final_estimator_.coef_[0], 2).tolist(),
    )
)
print(pl.DataFrame(rows, schema=["model", "train MCC", "test MCC", "meta weights"], orient="row"))

**Questions E1.**

1. Which meta weights are large in the leaking stack, and which in the out-of-fold stack? Which member does the leak
   over-trust, and why?
2. Does the leak change the test MCC a lot on the moons? Which of the three numbers (train, test, meta weights) shows
   the leak most clearly?
3. How close is your out-of-fold stack to `StackingClassifier`?

**How we got there.** `out_of_fold` is a cross-validation loop that keeps the predictions instead of the scores.
The members used at prediction time are refitted on all the training data, so the meta-learner sees at test time
the same kind of feature (probabilities of a model that did not see the point) that it saw in training.

**Answers E1.** 1-NN and the tree have perfect in-fold probabilities (they memorise the training set), so the leaking
meta-learner gives them the large weights; out-of-fold the weights move to the SVM, whose training predictions are honest.
The test MCC is 0.850 in both and the train MCC is 1.000 in both (the members are refitted on all the data); the meta
weights show the leak (2.98, 2.98, 1.66 against 0.94, 1.78, 2.87). The out-of-fold stack and `StackingClassifier` differ
only by the folds drawn.


# Part F — Challenge: Pseudo-Residuals with `jax.grad` (home)

For a loss other than the squared error, the tree is fitted to the **pseudo-residual**, the negative gradient of the loss
with respect to the current score. For the logistic loss of a score $F$ and a label $y \in \{0, 1\}$,
$\ell(F, y) = \ln(1 + e^{F}) - yF$, and the negative gradient is $y - \sigma(F)$.

## F1. Gradient boosting for classification

Implement `pseudo_residuals(y, F)` with `jax.grad` of the summed logistic loss (not the formula), and
`fit_gbc(X, y, n_trees, eta, depth)` returning `(F0, trees, eta)`: $F_0 = \ln\frac{\bar y}{1-\bar y}$, then in each round a
`DecisionTreeRegressor(max_depth=depth)` fitted to the pseudo-residuals, with $F \leftarrow F + \eta\,\hat f_b(x)$ (the leaf
values are not re-optimised: a simplification of the full algorithm). `predict_gbc_proba(model, X)` returns
$\sigma(F)$.

In [ ]:
def pseudo_residuals(y, F):
    def loss(F_):
        return jnp.sum(jnp.logaddexp(0.0, F_) - y * F_)

    return np.asarray(-jax.grad(loss)(jnp.asarray(F, dtype=jnp.float32)))


def fit_gbc(X, y, n_trees=100, eta=0.1, depth=3):
    f0 = float(np.log(y.mean() / (1 - y.mean())))
    F = np.full(len(y), f0)
    trees = []
    for _ in range(n_trees):
        r = pseudo_residuals(y, F)
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, r)
        F = F + eta * tree.predict(X)
        trees.append(tree)
    return f0, trees, eta


def predict_gbc_proba(model, X):
    f0, trees, eta = model
    F = f0 + eta * sum(t.predict(X) for t in trees)
    return 1 / (1 + np.exp(-F))

The driver checks the pseudo-residual against $y - \sigma(F)$, then fits 100 trees on the heart-disease data and compares
with `HistGradientBoostingClassifier` and with a random forest on the same split.

In [ ]:
Xh, yh = load_heart()
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, stratify=yh, random_state=0)
F_test = np.random.default_rng(0).normal(size=20)
y_small = (np.random.default_rng(1).random(20) > 0.5).astype(float)
print(
    "max |jax.grad - (y - sigma(F))| =",
    float(np.abs(pseudo_residuals(y_small, F_test) - (y_small - 1 / (1 + np.exp(-F_test)))).max()),
)
gbc = fit_gbc(Xh_tr, yh_tr, 100, 0.1, 3)
hgb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=0).fit(Xh_tr, yh_tr)
rf2 = RandomForestClassifier(300, random_state=0).fit(Xh_tr, yh_tr)
rows = [
    ("ours, 100 trees", f"{mcc(yh_te, predict_gbc_proba(gbc, Xh_te) > 0.5):.3f}"),
    ("HistGradientBoostingClassifier", f"{mcc(yh_te, hgb.predict(Xh_te)):.3f}"),
    ("RandomForestClassifier", f"{mcc(yh_te, rf2.predict(Xh_te)):.3f}"),
]
print(pl.DataFrame(rows, schema=["model", "test MCC"], orient="row"))
print(f"agreement ours / HistGB: {np.mean((predict_gbc_proba(gbc, Xh_te) > 0.5) == hgb.predict(Xh_te)):.1%}")

**Questions F1.**

1. How far is the `jax.grad` pseudo-residual from $y - \sigma(F)$, and why is it not exactly zero?
2. How close are the three models on 76 test patients (one patient moves MCC by about 0.03)? Can this split decide between them? Which
   protocol of Class 01 would you use?
3. Project 1: which learners (tree, KNN, network) would you put in a vote or a stack, and why?

**How we got there.** `jax.grad` of the summed loss gives the derivative with respect to each score, because the loss is a
sum of independent terms; the pseudo-residual is its negative.

**Answers F1.** The difference is about $5\times10^{-8}$: JAX computes in `float32`. The test MCC values are 0.658 (ours),
0.498 (HistGB) and 0.608 (forest), and ours agrees with HistGB on 89.5 % of the patients: our simplified version keeps the
leaf values of the trees and HistGB bins the features. On 76 patients one patient moves MCC by about 0.03, so one split
cannot decide; use repeated stratified cross-validation with the corrected $t$-test or a permutation test on the metric (demo E8).
For Project 1, members of different families (a tree, KNN, a network) make different errors, which is what a vote or a stack needs.
